In [1]:
import numpy as np
import nnfs
from nnfs.datasets import spiral_data

In [2]:
nnfs.init()

In [3]:
#Dense Layer
class Dense_Layer:
    #Layer Initialization
    def __init__(self,n_inputs,n_neurons,weights_regularizer_l1=0,biases_regularizer_l1=0,weights_regularizer_l2=0,biases_regularizer_l2=0):
        #Initialize the weights and biases
        #random.randn generates samples from the standard normal distribution (mean = 0, standard deviation = 1)
        
        self.weights=0.01*np.random.randn(n_inputs,n_neurons)

        #biases are initialized to zero and making it a 2D array with shape (1,n_neurons) 
        #so that it can be added to the output of the dot product of inputs and weights
        
        self.biases=np.zeros((1,n_neurons))

        #Set regularization strength for weights and biases
        #L1 regularization penalizes large absolute weight values and encourages some weights to become exactly or approximately zero.
        #L2 regularization discourages excessively large weights by penalizing their squared values. 
        #L2 generally encourages weights to remain small rather than forcing them to become zero.


        #Include regularization in the Dense_Layer.backward() method is that regularization doesn't just add a penalty to the loss.
        #It also needs to influence how the weights and biases are updated.
        self.weights_regularizer_l1=weights_regularizer_l1
        self.biases_regularizer_l1=biases_regularizer_l1
        self.weights_regularizer_l2=weights_regularizer_l2
        self.biases_regularizer_l2=biases_regularizer_l2

    #forward pass
    def forward(self,inputs):
        self.inputs=inputs
        #calculate the output values by the inputs,weights and biases
        self.output=np.dot(inputs,self.weights)+self.biases  
    #backward pass
    def backward(self,dvalues):
        #Gradients on parameters
        self.dweights=np.dot(self.inputs.T,dvalues)
        self.dbiases=np.sum(dvalues,axis=0,keepdims=True)
        
        #Gradient on regularization
        #L1 on weights
        if self.weights_regularizer_l1>0:
            dl1=np.ones_like(self.weights)
            dl1[self.weights<0]=-1
            self.dweights+=self.weights_regularizer_l1*dl1

        #L2 on weights
        if self.weights_regularizer_l2>0:
            self.dweights+=2*self.weights_regularizer_l2*self.weights

        #L1 on biases
        if self.biases_regularizer_l1>0:
            dl1=np.ones_like(self.biases)
            dl1[self.biases<0]=-1
            self.dbiases+=self.biases_regularizer_l1*dl1

        #L2 on biases
        if self.biases_regularizer_l2>0:
            self.dbiases+=2*self.biases_regularizer_l2*self.biases
             
        #Gradient on values
        self.dinputs=np.dot(dvalues,self.weights.T) 
     

In [4]:
class Dropout:
    def __init__(self,rate):
        #self.rate means “keep probability”
        self.rate=1-rate

    def forward(self,inputs,training):
        #inputs is typically shape (batch_size, n_neurons)
        self.inputs=inputs
        #Inference has no learning.There’s nothing for dropout’s noise to help with. 
        #Dropout only makes sense while the weights are changing. 
        #So the layer behaves differently depending on the phase, which is exactly what the training flag tells it.
        #training=True: drop and scale; training=False: pass through unchanged
        if training:
            self.binary_mask=np.random.binomial(1,self.rate,size=inputs.shape)/self.rate
            self.output=inputs*self.binary_mask
        else:
            self.binary_mask=np.ones_like(inputs)
            self.output=inputs.copy()
            

    def backward(self,dvalues):
        self.dinputs=dvalues*self.binary_mask

In [5]:
#Rectified Linear Unit (ReLU) Activation
class Activation_Relu:
    #Forward Pass
    def forward(self,inputs):
        #Calculate the output values from inputs
        ##Remember the inputs, backward needs them
        self.inputs=inputs
        self.output=np.maximum(0,inputs)
    #backward pass
    def backward(self,dvalues):
        #Since we need to modify the original variable, let's make a copy of the values first
        self.dinputs=dvalues.copy()
        #Zero gradient where input values were negative
        self.dinputs[self.inputs<=0]=0       

In [6]:
#SoftMax Activation
class Activation_Softmax:
    #Forward Pass
    def forward(self,inputs):
        #Get Unnormalized Probabilities
        #After subtracting the max, the largest value in each row becomes 0.
        #so the biggest exponent is e⁰ = 1 and everything else is between 0 and 1. No overflow

        exp_values=np.exp(inputs-np.max(inputs,axis=1,keepdims=True))
        
        #Normalize them for each sample
        probabilities=exp_values/np.sum(exp_values,axis=1,keepdims=True)
        #calculate the normalized probalities
        self.output=probabilities
    #backward pass
    def backward(self,dvalues):
        #create uninitialized array
        # The ​np.empty_like​ method creates an empty and uninitialized array. 
        # Uninitialized means that we can expect it to contain meaningless values, but we’ll set all of them shortly anyway
        self.dinputs=np.empty_like(dvalues)
        #enumerate outputs and gradients
        for index,(single_output,single_dvalues) in enumerate(zip(self.output,dvalues)):
            #flatten output array
            single_output=single_output.reshape(-1,1)
            #calculate Jacobian matrix of the output
            jacobian_matrix=np.diagflat(single_output)-np.dot(single_output,single_output.T)
            #calculate sample-wise gradient and add it to the array of sample gradients
            self.dinputs[index]=np.dot(jacobian_matrix,single_dvalues)
            #So these are the gradients of the loss with respect to Softmax’s  inputs for 1 sample.

In [7]:
class Sigmoid_Activation:
    def forward(self,inputs):
        #save inputs and calculate the output of the sigmoid function
        #The formula for the sigmoid function is: sigmoid(x) = 1 / (1 + e^(-x))
        self.inputs=inputs
        self.output=1/(1+np.exp(-inputs))
        
    def backward(self,dvalues):
        #The derivative of the sigmoid function is given by the formula: sigmoid'(x) = sigmoid(x) * (1 - sigmoid(x)).
        self.dinputs=dvalues*(1-self.output)*self.output


In [8]:
class Optimizer_SGD:
    def __init__(self,learning_rate=1,decay=0,momentum=0):

        #Stores the learning rate, which controls how big each step is. It defaults to 1
        #Iterations answers “how far along is training?” It starts at 0 and goes up by 1 every step, so the optimizer always knows what step it’s on.
        #Decay answers “how fast should the rate shrink as training goes on?” You pick it once and it never changes
        #Momentum answers “how much should we consider the previous update when making the current one?” You pick it once and it never changes

        self.learning_rate=learning_rate
        self.current_learning_rate=learning_rate
        self.decay=decay
        self.iteration=0

        #Momentum determines how much of the previous update contributes to the current update.
        self.momentum=momentum

    def pre_update_params(self):
        #if self.decay: means “only do this if decay is not zero
        #line is a shortcut for if self.decay != 0:
        if self.decay:
            self.current_learning_rate=self.learning_rate*(1/(1+self.decay*self.iteration))


    #Positive gradient: raising the weight raises the loss, so it goes down.
    #Negative gradient: raising the weight lowers the loss, so it goes up.
    
    def update_params(self,layer):
        #Layer contains the weights, biases, and their gradients. The optimizer updates them based on the gradients and learning rate.

        #Momentum enabled.
        if self.momentum:

            #The following code means:does this layer already have a variable called weight_momentums
            #The first iteration of training.There is no previous update yet.Therefore, we need to create an initial momentum value.

            if not hasattr(layer,'weight_momentums'):
            #if the layer does not contain momentum arrays, create them filled with zeros

                # every weights and biases needs its own momentum value.So we are creating an array of zeros with the same shape as the weights and biases.
                #This initialization happens only once for each layer. In subsequent iterations, the previous momentum arrays are reused.

                layer.weight_momentums=np.zeros_like(layer.weights)
                layer.bias_momentums=np.zeros_like(layer.biases)

            #build weight updates with momentum-take previous updates into account
            #update weight and bias momentums

            #The formula:----> current weights update = momentum * previous weights update - learning rate * current weight gradient
            #[momentum * previous weights update]--->This takes the previous weight update and multiplies it by the momentum coefficient.
            
            weight_updates=self.momentum*layer.weight_momentums-(self.current_learning_rate*layer.dweights)

            #we need to remember the latest update, not the update from two or three iterations ago.So we store the current update in the layer’s weight_momentums attribute.
            layer.weight_momentums=weight_updates

            #The formula:----> current biases update = momentum * previous biases update - learning rate * current bias gradient
            bias_updates=self.momentum*layer.bias_momentums-(self.current_learning_rate*layer.dbiases)
            layer.bias_momentums=bias_updates   
        
        #Momentum disabled: using ordinary SGD.
        else:
            #The formula:----> current weights update = - learning rate * current weight gradient
            weight_updates=-self.current_learning_rate*layer.dweights
            bias_updates=-self.current_learning_rate*layer.dbiases

        layer.weights+=weight_updates
        layer.biases+=bias_updates

    def post_update_params(self):
        self.iteration+=1
        

Momentum asks: How have I been moving, and can I use that movement to accelerate my progress?

Adagrad asks: How large have the historical gradients been for this particular parameter, and should I reduce its future update size?

*****The two learning-rate reductions for Adagrad Optimization
1. Global learning-rate decay
Reduces the learning rate as the number of iterations increases.Same reduction schedule for all parameters.

2. Adagrad's adaptive reduction
Reduces the learning rate according to each parameter's accumulated squared gradients.Different reduction for different parameters.

In [9]:
class Optimizer_Adagrad:
    #Adagrad doesn't have a momentum coefficient.
    #It has a decay rate, which is the same as in SGD.
    #It also has an epsilon value, which is a small number that prevents division by zero when normalizing the gradients.
    

    def __init__(self,learning_rate=1,decay=0,epsilon=1e-7):
        self.learning_rate=learning_rate
        self.current_learning_rate=learning_rate
        self.decay=decay
        self.iteration=0
        self.epsilon=epsilon

    def pre_update_params(self):

        #This decay is not unique to Adagrad. 
        #It is a general learning-rate scheduling mechanism that your optimizer implements separately from Adagrad's adaptive mechanism.
        
        if self.decay:

            #It gradually reduces the learning rate as training progresses.
            #The formula for the decayed learning rate is: current_learning_rate = initial_learning_rate * (1 / (1 + decay * iteration number))
            
            self.current_learning_rate=self.learning_rate*(1/(1+self.decay*self.iteration))

    def update_params(self,layer):

        #Momentum remembers previous parameter updates.
        #Adagrad remembers the sum of previous squared gradients.
        #If the layer does not contain cache arrays, create them filled with zeros.
        #Every weights and biases needs its own momentum value.
        #So we are creating an array of zeros with the same shape as the weights and biases.

        if not hasattr(layer,'weight_cache'):
            layer.weight_cache=np.zeros_like(layer.weights)
            layer.bias_cache=np.zeros_like(layer.biases)

        #Update cache with squared current gradients.
        #Unlike momentum, Adagrad does not reduce the influence of older gradients.It accumulates the sum of squared gradients over time. 
        #This means that as training progresses, the cache values grow larger, which in turn reduces the effective learning rate for parameters that have received large gradients in the past.
        
        layer.weight_cache+=layer.dweights**2
        layer.bias_cache+=layer.dbiases**2

        # Vanilla SGD parameter update + normalization with square rooted cache
        layer.weights+=-self.current_learning_rate*layer.dweights/(np.sqrt(layer.weight_cache)+self.epsilon)
        layer.biases+=-self.current_learning_rate*layer.dbiases/(np.sqrt(layer.bias_cache)+self.epsilon)

    def post_update_params(self):
        self.iteration+=1

In [10]:
class Optimizer_RMSprop:
    #RMSprop is similar to Adagrad, but it introduces a decay factor for the cache, which helps to prevent the learning rate from decreasing too quickly.
    #It has a decay rate, which is the same as in SGD.
    #It also has an epsilon value, which is a small number that prevents division by zero when normalizing the gradients.
    ##The formula is---> C_t=rho* C_{t-1}+(1-\rho)*g_t^2
    #This formula gives more importance to recent gradients and gradually reduces the influence of older gradients.

    def __init__(self,learning_rate=0.001,decay=0,rho=0.9,epsilon=1e-7):
        self.learning_rate=learning_rate
        self.current_learning_rate=learning_rate
        self.decay=decay
        self.iteration=0
        self.rho=rho
        self.epsilon=epsilon

    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate=self.learning_rate*(1/(1+self.decay*self.iteration))

    def update_params(self,layer):
        if not hasattr(layer,'weight_cache'):
            layer.weight_cache=np.zeros_like(layer.weights)
            layer.bias_cache=np.zeros_like(layer.biases)

        # Update cache with squared current gradients and apply decay factor
        layer.weight_cache=self.rho*layer.weight_cache+(1-self.rho)*layer.dweights**2
        layer.bias_cache=self.rho*layer.bias_cache+(1-self.rho)*layer.dbiases**2

        # Vanilla SGD parameter update + normalization with square rooted cache
        layer.weights+=-self.current_learning_rate*layer.dweights/(np.sqrt(layer.weight_cache)+self.epsilon)
        layer.biases+=-self.current_learning_rate*layer.dbiases/(np.sqrt(layer.bias_cache)+self.epsilon)

    def post_update_params(self):
        self.iteration+=1

In [11]:
class Optimizer_Adam:

    #Adam combines the benefits of both momentum and Adagrad. 
    #The main idea: Adam combines two ideas you've already learned:
    #1. Momentum: remembers the direction of previous gradients.
    #2. RMSprop: remembers the magnitude of previous squared gradients and uses that information to adjust the learning rate.

    def __init__(self,learning_rate=0.001,decay=0,epsilon=1e-7,beta_1=0.9,beta_2=0.999):
        self.learning_rate=learning_rate
        self.current_learning_rate=learning_rate
        self.decay=decay
        self.iteration=0
        self.epsilon=epsilon

        #beta_1	Controls how much Adam remembers previous gradients (Momentum).
        self.beta_1=beta_1
        
        #beta_2	Controls how much Adam remembers previous squared gradients (RMSprop).
        self.beta_2=beta_2

    def pre_update_params(self):

        #Same learning-rate decay mechanism we have seen in SGD, Adagrad, and RMSprop.
        if self.decay:
            self.current_learning_rate=self.learning_rate*(1/(1+self.decay*self.iteration))

    def update_params(self,layer):
        if not hasattr(layer,'weight_cache'):

            #Adam needs four separate arrays
            #weight_momentums stores momentum for weights.
            #weight_cache stores squared-gradient information for weights.
            #bias_momentums	stores momentum for biases.
            #bias_cache	stores squared-gradient information for biases.

            layer.weight_momentums=np.zeros_like(layer.weights)
            layer.weight_cache=np.zeros_like(layer.weights)
            layer.bias_momentums=np.zeros_like(layer.biases)
            layer.bias_cache=np.zeros_like(layer.biases)

        # Update momentum with current gradients
        #This is the Momentum component
        layer.weight_momentums=self.beta_1*layer.weight_momentums+(1-self.beta_1)*layer.dweights
        layer.bias_momentums=self.beta_1*layer.bias_momentums+(1-self.beta_1)*layer.dbiases

        # Get corrected momentum
        weight_momentums_corrected=layer.weight_momentums/(1-self.beta_1**(self.iteration+1))
        bias_momentums_corrected=layer.bias_momentums/(1-self.beta_1**(self.iteration+1))

        # Update cache with squared current gradients
        #This is the RMSprop component
        layer.weight_cache=self.beta_2*layer.weight_cache+(1-self.beta_2)*layer.dweights**2
        layer.bias_cache=self.beta_2*layer.bias_cache+(1-self.beta_2)*layer.dbiases**2

        # Get corrected cache
        weight_cache_corrected=layer.weight_cache/(1-self.beta_2**(self.iteration+1))
        bias_cache_corrected=layer.bias_cache/(1-self.beta_2**(self.iteration+1))

        # Vanilla SGD parameter update + normalization
        layer.weights-=self.current_learning_rate*weight_momentums_corrected/(np.sqrt(weight_cache_corrected)+self.epsilon)
        layer.biases-=self.current_learning_rate*bias_momentums_corrected/(np.sqrt(bias_cache_corrected)+self.epsilon)

    def post_update_params(self):
        self.iteration+=1

In [12]:
#Common Loss Function Class
class Loss:

    #calculate two things----->
    # 1. How far are the model's predictions from the actual answers? 
    # 2. How much are the weights and biases being penalized for being too large?

    #Calculate the data and regularization losses
    #Given model output and ground truth values

    def regularization_loss(self,layer):
        #0 by default
        regularization_loss=0
        if layer.weights_regularizer_l1 > 0:
            regularization_loss += layer.weights_regularizer_l1 * np.sum(np.abs(layer.weights))
        if layer.biases_regularizer_l1 > 0:
            regularization_loss += layer.biases_regularizer_l1 * np.sum(np.abs(layer.biases))
        if layer.weights_regularizer_l2 > 0:
            regularization_loss += layer.weights_regularizer_l2 * np.sum(layer.weights ** 2)
        if layer.biases_regularizer_l2 > 0:
            regularization_loss += layer.biases_regularizer_l2 * np.sum(layer.biases ** 2)
        return regularization_loss

    
    #This method calculates the average data loss

    def calculate(self,output,y):
        #Calculate the sample loss
        #This Loss class doesn't define its own forward() method. Instead, a child class(loss_CategoricalCrossentropy) implements it.
        sample_losses=self.forward(output,y)
        #calculate the mean loss
        data_loss=np.mean(sample_losses)
        return data_loss

In [13]:
#Cross Entropy Loss for Categorical Classification
class Loss_CategoricalCrossentropy(Loss):
    #forward Pass
    def forward(self,y_pred,y_true):
        #restrict (limit) the values of y_prediction within a specified minimum and maximum range
        y_pred_clipped=np.clip(y_pred,1e-7,1-1e-7)
        #for categorical labels
        if len(y_true.shape)==1:
            correct_confidences=y_pred_clipped[range(len(y_pred)),y_true]
        elif len(y_true.shape)==2:
            product=y_pred_clipped*y_true
            correct_confidences=np.sum(product,axis=1)
        neg_log_likelihood=-np.log(correct_confidences)
        return neg_log_likelihood  
        
    #Backward Pass

    #It computes how much the loss changes w.r.t each predicted probability, so that gradient can flow back into the earlier layers.
    #the loss is L = -Σ y_true[i] · log(y_pred[i]). 
    #Its derivative with respect to each predicted probability is -y_true[i] / y_pred[i]

    def backward(self,y_pred,y_true):

        #number of samples
        #samples = len() gives the batch size
        
        samples=len(y_pred)
        
        #number of labels in every sample
        labels=len(y_pred[0])

        #y_true can come in two forms: sparse class indices like [0, 2, 1] (1D), or one-hot vectors (2D). 
        #The formula needs one-hot, so if it’s 1D, it converts it. np.eye(labels) builds an identity matrix
        #  Thus indexing it with y_true picks one row per sample.

        if len(y_true.shape)==1:
            y_true=np.eye(labels)[y_true]
        
        # calculate the gradient
        # dinputs answers the question: if I nudge each predicted probability a little, how much does the loss change?
        # y_true is one-hot, everything is 0 except at the true class. 
        # For a prediction [0.7, 0.2, 0.1] with true class 0, the result is [-1/0.7, 0, 0] ≈ [-1.43, 0, 0]. 
        # The negative sign means “increase this probability to lower the loss”, and the smaller the predicted probability, the stronger the push.
        
        self.dinputs=-y_true/y_pred
        
        #normalize the gradient
        #The loss you report is the mean over the batch, so its gradient gets a 1/samples factor. 
        # This also keeps the gradient size independent of batch size, which makes the learning rate behave consistently.

        self.dinputs=self.dinputs/samples     

In [14]:
class Activation_Softmax_Loss_CategoricalCrossentropy():
    #create activation and loss function objects
    def __init__(self):
        self.activation=Activation_Softmax()
        self.loss=Loss_CategoricalCrossentropy()
    #forward pass
    def forward(self,inputs,y_true):
        #output layer's activation function
        self.activation.forward(inputs)
        #set the output
        self.output=self.activation.output
        #calculate and return loss value
        return self.loss.calculate(self.output,y_true)
    #backward pass
    def backward(self,dvalues,y_true):
        #number of samples in a batch
        samples=len(dvalues)

        #if labels are one-hot encoded, turn them into discrete values
        #This makes sure y_true is always a 1D array of class numbers, no matter which label format we started with.
        #y_true.shape is a tuple of the array’s dimensions, and len() of it tells you how many dimensions there are
        
        if len(y_true.shape)==2:
            y_true=np.argmax(y_true,axis=1)
        #copy so we can safely modify

        self.dinputs=dvalues.copy()
        
        #calculate gradient
        #samples supplies the row numbers, y_true supplies the column numbers.
        # together they point at one cell per row, the true class, which is where ŷ − y differs from ŷ.
        #the gradient of softmax + cross-entropy is ŷ − y, and y is a one-hot vector 
        #so “minus y” means “minus 1 at the true class, minus 0 everywhere else.

        self.dinputs[range(samples),y_true]-=1
        
        #normalize gradient by mean of samples
        # This ensures that the gradient is averaged across all samples in the batch, making it independent of the batch size.
        
        self.dinputs=self.dinputs/samples
        #backward doesn’t return anything. It saves its result as an attribute.

In [15]:
#Cross Entropy Loss for Binary Classification
class Loss_BinaryCrossentropy(Loss):
    def forward(self,y_pred,y_true):
        #restrict (limit) the values of y_prediction within a specified minimum and maximum range
        y_pred_clipped=np.clip(y_pred,1e-7,1-1e-7)
        #calculate sample-wise loss
        sample_losses=-(y_true*np.log(y_pred_clipped)+(1-y_true)*np.log(1-y_pred_clipped))
        sample_losses=np.mean(sample_losses,axis=-1)
        return sample_losses

    def backward(self,dvalues,y_true):
        #number of samples
        samples=len(dvalues)
        #number of outputs in every sample
        outputs=len(dvalues[0])
        #clip data to prevent division by 0
        clipped_dvalues=np.clip(dvalues,1e-7,1-1e-7)
        #calculate gradient
        self.dinputs=-(y_true/clipped_dvalues-(1-y_true)/(1-clipped_dvalues))/outputs
        #normalize gradient
        self.dinputs=self.dinputs/samples

In [16]:
nnfs.init()
X,y=spiral_data(samples=100,classes=2)

In [17]:
y=y.reshape(-1,1)

In [18]:
print(y[0:5])

[[0]
 [0]
 [0]
 [0]
 [0]]


In [19]:
dense1=Dense_Layer(n_inputs=2,n_neurons=64,weights_regularizer_l2=5e-4,biases_regularizer_l2=5e-4)

In [20]:
activation1=Activation_Relu()

In [21]:
#dropout1=Dropout(rate=0.1)

In [22]:
dense2=Dense_Layer(n_inputs=64,n_neurons=1)

In [23]:
activation2=Sigmoid_Activation()

In [24]:
loss_function=Loss_BinaryCrossentropy()

In [25]:
optimizer=Optimizer_Adam(decay=5e-7)

In [26]:
for epoch in range(10001):
    dense1.forward(X)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)
    activation2.forward(dense2.output)
    data_loss=loss_function.calculate(activation2.output,y)
    regularization_loss=loss_function.regularization_loss(dense1)+loss_function.regularization_loss(dense2)
    loss=data_loss+regularization_loss
    predictions=(activation2.output>0.5)*1
    accuracy=np.mean(predictions==y)
    if not epoch%100:
        print(f'epoch: {epoch}, '+
              f'acc: {accuracy:.3f}, '+
              f'loss: {loss:.3f} ('+
              f'data_loss: {data_loss:.3f}, '+
              f'reg_loss: {regularization_loss:.3f}), '+
              f'lr: {optimizer.current_learning_rate}')
    #Backpropagation    
    loss_function.backward(activation2.output,y)
    activation2.backward(loss_function.dinputs)
    dense2.backward(activation2.dinputs)
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    #Update weights and biases
    optimizer.pre_update_params()
    optimizer.update_params(dense1)
    optimizer.update_params(dense2)
    optimizer.post_update_params()


epoch: 0, acc: 0.500, loss: 0.693 (data_loss: 0.693, reg_loss: 0.000), lr: 0.001
epoch: 100, acc: 0.630, loss: 0.674 (data_loss: 0.673, reg_loss: 0.001), lr: 0.0009999505024501287
epoch: 200, acc: 0.625, loss: 0.669 (data_loss: 0.668, reg_loss: 0.001), lr: 0.0009999005098992651
epoch: 300, acc: 0.645, loss: 0.665 (data_loss: 0.663, reg_loss: 0.002), lr: 0.000999850522346909
epoch: 400, acc: 0.650, loss: 0.659 (data_loss: 0.657, reg_loss: 0.002), lr: 0.0009998005397923115
epoch: 500, acc: 0.675, loss: 0.648 (data_loss: 0.644, reg_loss: 0.004), lr: 0.0009997505622347225
epoch: 600, acc: 0.720, loss: 0.632 (data_loss: 0.626, reg_loss: 0.006), lr: 0.0009997005896733929
epoch: 700, acc: 0.770, loss: 0.614 (data_loss: 0.604, reg_loss: 0.010), lr: 0.0009996506221075735
epoch: 800, acc: 0.775, loss: 0.594 (data_loss: 0.579, reg_loss: 0.015), lr: 0.000999600659536515
epoch: 900, acc: 0.775, loss: 0.575 (data_loss: 0.556, reg_loss: 0.020), lr: 0.0009995507019594694
epoch: 1000, acc: 0.785, loss:

In [27]:
X_test,y_test=spiral_data(samples=100,classes=2)
y_test=y_test.reshape(-1,1)
dense1.forward(X_test)

In [28]:
activation1.forward(dense1.output)

In [29]:
dense2.forward(activation1.output)

In [30]:
activation2.forward(dense2.output)
loss=loss_function.calculate(activation2.output,y_test)

In [31]:
predictions=(activation2.output>0.5)*1
accuracy=np.mean(predictions==y_test)
print(f'validation, acc: {accuracy:.3f}, loss: {loss:.3f}')

validation, acc: 0.920, loss: 0.237
